# Colab: Smoke + Full SFT + Full GRPO

Run setup first. Smoke is only for debugging. Full cycle is SFT -> GRPO -> checkpoint eval.


In [ ]:
# 1. Fresh clone + dependencies.
%cd /content
!rm -rf /content/ml_project
!git clone https://github.com/Andrii238/ml-project.git /content/ml_project
%cd /content/ml_project
!pip install -q -r requirements-colab.txt
!pip uninstall -y -q torchao torchvision bitsandbytes

import torch
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
import transformers, trl, peft
print('transformers', transformers.__version__, '| trl', trl.__version__, '| peft', peft.__version__)


In [ ]:
# 2. Smoke full pipeline: small SFT -> short GRPO -> eval.
# Purpose: catch crashes/reward/parsing/checkpoint bugs before the full run.
!rm -rf ckpts/smoke_sft ckpts/smoke_grpo
!PYTHONPATH=/content/ml_project PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True   python notebooks/smoke_full_pipeline.py --clean --sft-epochs 1 --grpo-steps 12 --n-val 8 --samples-per-layout 1


In [ ]:
# 3. Full SFT only. Run this after smoke passes.
!rm -rf ckpts/sft
!PYTHONPATH=/content/ml_project PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True   python -m training.train_sft --output-dir ./ckpts/sft --num-train-epochs 1


In [ ]:
# 4. Full SFT smoke eval.
!PYTHONPATH=/content/ml_project python notebooks/eval_sft_smoke.py

import json, os
path = 'results/eval_sft_vs_base_smoke.json'
print('exists:', os.path.exists(path), path)
if os.path.exists(path):
    print(json.dumps(json.load(open(path)), indent=2))


In [ ]:
# 5. Full GRPO. Saves checkpoints at 25, 50, 75, and final 100.
!rm -rf ckpts/grpo
!PYTHONPATH=/content/ml_project PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  python -m training.train_grpo \
  --init-adapter ./ckpts/sft \
  --curriculum \
  --output-dir ./ckpts/grpo \
  --group-size 8 \
  --temperature 1.0 \
  --learning-rate 5e-5 \
  --per-device-batch-size 2 \
  --gradient-accumulation-steps 4 \
  --max-steps 100 \
  --save-steps 25 \
  --max-prompt-length 3500 \
  --max-completion-length 512


In [ ]:
# 6. Final deterministic checkpoint eval. This is the main result table.
import os, json, pandas as pd
os.makedirs('results', exist_ok=True)
from training.evaluate import evaluate_checkpoints, save_results, rows_to_table
results = evaluate_checkpoints(
    [{'name': 'policy_0', 'adapter': None},
     {'name': 'policy_1_sft', 'adapter': './ckpts/sft'},
     {'name': 'policy_2_grpo25', 'adapter': './ckpts/grpo/checkpoint-25'},
     {'name': 'policy_3_grpo50', 'adapter': './ckpts/grpo/checkpoint-50'},
     {'name': 'policy_4_grpo75', 'adapter': './ckpts/grpo/checkpoint-75'},
     {'name': 'policy_final', 'adapter': './ckpts/grpo'}],
    samples_per_layout=1,
    n_val=40,
)
print(rows_to_table(results))
save_results(results, 'results/eval_final.json')
d = json.load(open('results/eval_final.json'))
rows = [{'ckpt': c['name'], 'composite': round(c['mean_composite'], 4),
         'green_sci/s': round(c['mean_green_science'], 4),
         'valid %': round(c['valid_output_pct'], 1),
         'parse_ok %': round(c['parse_ok_pct'], 1),
         'materials': round(c['mean_materials'], 1),
         'cells': round(c['mean_cells'], 1),
         'machines': round(c['mean_machines'], 2)} for c in d]
pd.DataFrame(rows).set_index('ckpt')


In [ ]:
# 7. Full GRPO progress diagnostic. No training here; evaluates existing checkpoints.
!PYTHONPATH=/content/ml_project python notebooks/debug_grpo_progress.py \
  --sft-dir ./ckpts/sft \
  --grpo-dir ./ckpts/grpo \
  --n-val 8 \
  --sampled-per-layout 4 \
  --max-new-tokens 512 \
  --out results/grpo_progress_full.json
